# End-to-End Seismic Pipeline: 2019 Ridgecrest Aftershock Sequence

This notebook runs the **Seismic Workflow Engine (SWE)** pipeline on data from the 2019 Ridgecrest, California aftershock sequence.

**Dataset**: 2 hours of broadband vertical (BHZ, ~40 Hz) data from 5 CI-network stations near Ridgecrest, recorded 2019-07-06 04:00-06:00 UTC - immediately following the M7.1 mainshock. This window contains dozens of M2+ aftershocks per hour, making it ideal for demonstrating automated detection workflows.

**Pipeline steps**:
1. Data acquisition via FDSN
2. Waveform & spectral visualization
3. Noise characterization (STA/LTA)
4. Deep-learning phase picking (SeisBench PhaseNet)
5. Phase-to-event association (DBSCAN)
6. Matched-filter detection (EQcorrscan)
7. Grid-search hypocenter location

> **Requirements**: `pip install -e .[picking]` for SeisBench support. Internet access required for FDSN data download.

In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.dates import num2date
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

import swe.io
import swe.noise
import swe.association
import swe.location
import swe.viz
import swe.utils

# SeisBench is an optional dependency
try:
    import swe.picking
    HAS_SEISBENCH = True
except ImportError:
    HAS_SEISBENCH = False
    print(
        "SeisBench not installed - phase-picking cells will be skipped.\n"
        "Install with: pip install seisbench"
    )

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

%matplotlib inline

In [ ]:
# ---- Configuration --------------------------------------------------------
NETWORK = "CI"
STATIONS = ["WBS", "CLC", "SRT", "JRC2", "TOW2"]
CHANNELS = "BHZ"
START = UTCDateTime("2019-07-06T04:00:00")
END = UTCDateTime("2019-07-06T06:00:00")

DATA_DIR = Path("data/ridgecrest")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Acquisition

We download waveforms from the IRIS FDSN service using `swe.io.get_waveforms()`, which writes daily MiniSEED files to disk (needed later by the noise module). We also fetch station metadata to build a coordinate dictionary for the association and location steps.

In [ ]:
# Download waveforms to disk (MiniSEED)
mseed_files = swe.io.get_waveforms(
    NETWORK, STATIONS, CHANNELS, START, END,
    output_dir=DATA_DIR,
    client="IRIS",
)
print(f"Downloaded {len(mseed_files)} file(s): {[p.name for p in mseed_files]}")

# Also load everything into a single in-memory Stream for picking / viz
import obspy
st = obspy.Stream()
for f in mseed_files:
    st += obspy.read(str(f))
st.merge(method=1, fill_value=0)

# Fetch station coordinates dynamically
client = Client("IRIS")
inventory = client.get_stations(
    network=NETWORK, station=",".join(STATIONS), channel=CHANNELS,
    starttime=START, endtime=END, level="station",
)

location_dict: dict[str, list[float]] = {}
for net in inventory:
    for sta in net:
        location_dict[sta.code] = [sta.longitude, sta.latitude, sta.elevation / 1000.0]

print(f"\nStation coordinates (lon, lat, elev_km):")
for name, coords in location_dict.items():
    print(f"  {name}: {coords[0]:.4f}, {coords[1]:.4f}, {coords[2]:.3f}")

In [ ]:
# Verify data quality
print(st)
print(f"\nTraces: {len(st)}")
for tr in st:
    print(
        f"  {tr.id}  {tr.stats.starttime} - {tr.stats.endtime}  "
        f"({tr.stats.sampling_rate} Hz, {tr.stats.npts} samples)"
    )

## 2. Waveform Visualization

SWE provides multi-panel waveform plots, spectrograms, and cross-station spectral comparisons. These help assess data quality and identify the frequency content of aftershocks.

In [ ]:
# Multi-panel overview of the full stream
fig = swe.viz.plot_stream(st)
fig.set_size_inches(14, 8)
plt.tight_layout()
plt.show()

In [ ]:
# Spectrogram for the first station - look for aftershock energy bursts
fig = swe.viz.plot_spectrogram(st[0], freq_max=20.0)
plt.show()

In [ ]:
# Cross-station spectral comparison
fig = swe.viz.plot_multi_spectra(st)
plt.show()

## 3. Noise Characterization

The noise module uses STA/LTA ratios to identify quiescent windows in continuous data. During an active aftershock sequence we expect **very few** noise windows - most of the record is dominated by seismic signals.

**STA/LTA parameters** (tuned for 40 Hz BHZ data):
- STA = 20 samples (0.5 s), LTA = 200 samples (5 s)
- Threshold = 2.5 (windows below this ratio are "noise")
- Window length = 800 samples (20 s)

In [ ]:
# STA/LTA noise detection
detection_params = (20, 200, 2.5, 800)  # (sta, lta, threshold, window_length)
time_periods = [(START, END)]

# The default parser now handles both "NETWORK.STATION..CHANNEL.DATE.ms"
# and "NETWORK.STATION.CHANNEL.DATE.ms" formats automatically.
noise_windows, noise_times = swe.noise.find_noise_windows(
    [str(f) for f in mseed_files],
    detection_params,
    time_periods,
)

print(f"Noise windows found: {len(noise_times)}")

In [ ]:
# Per-station noise metrics (if any windows were found)
if len(noise_times) > 0:
    metrics = swe.noise.analyze_noise_windows(noise_windows, noise_times)
    for station, station_metrics in metrics.items():
        print(f"  {station}: {len(station_metrics)} window(s)")
        for m in station_metrics:
            print(f"    max={m[0]:.2f}, median={m[1]:.2f}, rms={m[2]:.2f}")
else:
    print(
        "No noise windows found - expected during an active aftershock sequence.\n"
        "The STA/LTA ratio stays above the threshold because seismic energy is\n"
        "nearly continuous in this 2-hour post-mainshock window."
    )

In [ ]:
# Noise histograms (only if we found noise windows)
if len(noise_times) > 0:
    fig = swe.noise.plot_noise_time_histogram(
        noise_times, bin_width=600,
        major_x_tick="hour", minor_x_tick="minute",
    )
    plt.show()

    fig = swe.noise.plot_noise_station_histogram(noise_times, STATIONS)
    plt.show()
else:
    print("Skipping noise histograms - no noise windows to plot.")

## 4. Phase Picking (SeisBench)

[PhaseNet](https://doi.org/10.1093/gji/ggy423) is a deep-learning picker trained on Southern California earthquake data - an excellent match for Ridgecrest BHZ waveforms. SWE wraps SeisBench models via `swe.picking`.

> **Note**: SeisBench is an optional dependency. The first run downloads ~150 MB of model weights. If unavailable, this section is skipped.

In [ ]:
if HAS_SEISBENCH:
    # Detailed picks with confidence scores
    detailed_picks = swe.picking.predict_phase_arrivals(
        st, model_name="PhaseNet", p_threshold=0.3, s_threshold=0.3,
    )

    print(f"Total picks: {len(detailed_picks)}")
    # Summary by station and phase
    from collections import Counter
    station_counts = Counter(p.station for p in detailed_picks)
    phase_counts = Counter(p.phase for p in detailed_picks)
    print(f"By station: {dict(station_counts)}")
    print(f"By phase:   {dict(phase_counts)}")
else:
    print("SeisBench not available - skipping phase picking.")

In [ ]:
if HAS_SEISBENCH:
    # Station-grouped picks for the association step
    station_picks = swe.picking.pick_stream(
        st, model_name="PhaseNet", p_threshold=0.3, s_threshold=0.3,
    )

    for sta, picks in station_picks.items():
        print(f"  {sta}: {len(picks)} picks")
else:
    station_picks = None
    print("SeisBench not available - skipping.")

In [ ]:
if HAS_SEISBENCH and detailed_picks:
    # Pick confidence distribution
    confidences = [p.confidence for p in detailed_picks]
    fig = swe.viz.plot_distribution(
        confidences, bins=50, title="Pick confidence distribution", xlabel="Confidence",
    )
    plt.show()

    # Show picks on a 1-minute waveform window around the first high-confidence pick
    best_picks = sorted(detailed_picks, key=lambda p: -p.confidence)
    ref_time = best_picks[0].time
    st_window = st.copy().trim(ref_time - 30, ref_time + 30)

    # Build per-trace pick list for plot_event_picks
    window_picks = []
    for tr in st_window:
        sta = tr.stats.station
        matching = [
            p for p in detailed_picks
            if p.station == sta
            and tr.stats.starttime <= p.time <= tr.stats.endtime
        ]
        if matching:
            window_picks.append(matching[0])
        else:
            window_picks.append(swe.picking.PhaseArrival(
                station=sta, phase="P", time=ref_time, confidence=0.0,
            ))

    # Convert PhaseArrival NamedTuples to dicts (plot_event_picks expects dicts)
    fig = swe.viz.plot_event_picks(
        st_window,
        [{"time": p.time} for p in window_picks],
    )
    plt.show()
else:
    if not HAS_SEISBENCH:
        print("SeisBench not available - skipping pick visualization.")

## 5. Phase Association

The association module clusters picks from multiple stations into coherent events using DBSCAN. Key parameters:
- `dbscan_eps=0.5`: neighborhood radius for clustering
- `dbscan_min_samples=4`: minimum picks to form an event
- `temporal_threshold=2.0`: max seconds from cluster median
- `spatial_threshold=5.0`: max station-distance difference

In [ ]:
if HAS_SEISBENCH and station_picks:
    station_list = [sta for sta in STATIONS if sta in station_picks]

    result = swe.association.associate_picks(
        station_picks,
        station_list,
        location_dict,
        dbscan_eps=0.5,
        dbscan_min_samples=4,
        temporal_threshold=2.0,
        spatial_threshold=5.0,
        sampling_rate=st[0].stats.sampling_rate,
    )

    station_distance = result.station_distance
    reverse_station_distance = result.reverse_station_distance
    pick_station_order = result.pick_station_order
    pick_times = result.pick_times
    labels = result.labels

    unique_labels, cluster_medians = swe.association.get_cluster_stats(labels, pick_times)

    n_events = len([l for l in unique_labels if l != -1])
    n_noise = int(np.sum(labels == -1))
    print(f"Events found: {n_events}")
    print(f"Noise picks:  {n_noise}")
    print(f"\nEvent times (UTC):")
    for lbl in sorted(unique_labels):
        if lbl == -1:
            continue
        median_date = num2date(cluster_medians[lbl])
        n_picks = int(np.sum(labels == lbl))
        print(f"  Event {lbl}: {median_date:%Y-%m-%d %H:%M:%S} ({n_picks} picks)")
else:
    labels = None
    print("No picks available - skipping association.")

In [ ]:
# Cluster statistics
if labels is not None and n_events > 0:
    picks_per_event = [
        int(np.sum(labels == lbl))
        for lbl in unique_labels if lbl != -1
    ]
    print(f"Picks per event: min={min(picks_per_event)}, "
          f"max={max(picks_per_event)}, median={np.median(picks_per_event):.0f}")

    # Event rate
    event_times_utc = [
        num2date(cluster_medians[lbl])
        for lbl in sorted(unique_labels) if lbl != -1
    ]
    duration_min = (END - START) / 60.0
    print(f"Event rate: {n_events / (duration_min / 60.0):.1f} events/hour "
          f"over {duration_min:.0f} minutes")

## 6. Matched-Filter Detection

We use the best-populated association cluster as a template to scan the full 2-hour record for additional similar events via cross-correlation (matched-filter). This is handled by [EQcorrscan](https://eqcorrscan.readthedocs.io/).

Steps:
1. Select the best cluster (most picks) from the association output
2. Build an ObsPy `Event` with `Pick` objects from the cluster
3. Construct an EQcorrscan `Tribe` (template library)
4. Run `Tribe.detect()` on the 2-hour stream

In [ ]:
if labels is not None and n_events > 0:
    from eqcorrscan import Tribe

    # Build ObsPy Catalog directly from association output
    catalog = swe.association.association_to_catalog(
        result,
        network=NETWORK,
        channel=CHANNELS,
        phase_hint="P",
        default_latitude=35.77,
        default_longitude=-117.60,
        default_depth=8000.0,
    )

    # Select the best (most-picked) event as template
    best_event = max(catalog.events, key=lambda e: len(e.picks))
    print(f"Template event: {best_event.origins[0].time}, {len(best_event.picks)} picks")
    print(f"Built catalog with {len(catalog)} events")

    # Derive best_indices and best_origin_time for use by the location cell
    best_label = max(
        [l for l in unique_labels if l != -1],
        key=lambda l: int(np.sum(labels == l)),
    )
    best_indices = np.where(labels == best_label)[0]
    best_origin_time = UTCDateTime(num2date(cluster_medians[best_label]))
else:
    catalog = None
    print("No association results - skipping template construction.")

In [ ]:
if catalog is not None:
    # Prepare stream for template construction
    st_proc = st.copy()
    st_proc.filter("bandpass", freqmin=1.0, freqmax=15.0, corners=4)
    st_proc.resample(40.0)  # ensure consistent sampling rate

    # Construct tribe from catalog
    tribe = Tribe().construct(
        method="from_meta_file",
        meta_file=catalog,
        st=st_proc,
        lowcut=1.0,
        highcut=15.0,
        samp_rate=40.0,
        filt_order=4,
        length=6.0,
        prepick=0.5,
        swin="all",
        process_len=7200,  # 2-hour window
    )
    print(f"Tribe: {len(tribe)} template(s), "
          f"{sum(len(t.st) for t in tribe)} trace(s) total")
else:
    tribe = None
    print("No catalog - skipping tribe construction.")

In [ ]:
if tribe is not None and len(tribe) > 0:
    # Run matched-filter detection on the 2-hour window
    # Use daylong=False since we have a sub-daily record
    party = tribe.detect(
        stream=st_proc,
        threshold=8.0,
        threshold_type="MAD",
        trig_int=6.0,
        daylong=False,
        process_length=7200,
    )

    detections = [d for family in party for d in family]
    print(f"Matched-filter detections: {len(detections)}")
    for d in detections[:10]:
        print(f"  {d.detect_time}  cc={d.detect_val:.3f}")
    if len(detections) > 10:
        print(f"  ... and {len(detections) - 10} more")
else:
    detections = []
    print("No templates - skipping matched-filter detection.")

In [ ]:
if detections:
    detection_times = [d.detect_time for d in detections]
    fig = swe.viz.plot_date_distribution(
        detection_times, bins=30, title="Matched-filter detections over time",
    )
    plt.show()
else:
    print("No detections to plot.")

## 7. Hypocenter Location

SWE includes two grid-search location methods that work without external software:
- **`locate_by_travel_time`**: Constant-velocity travel-time inversion
- **`amplocate`**: Amplitude-decay inversion

We convert station coordinates from geographic (lat/lon) to a local Cartesian frame (meters) with the network centroid as origin.

In [ ]:
# Convert station coordinates to local Cartesian (meters) using geo_to_local
active_stations = [s for s in STATIONS if s in location_dict]
active_location_dict = {s: location_dict[s] for s in active_stations}

local_coords, (origin_lon, origin_lat, origin_elev) = swe.utils.geo_to_local(active_location_dict)
print(f"Network centroid: {origin_lon:.4f}°E, {origin_lat:.4f}°N, {origin_elev:.3f} km")

# Build station_coords array for the location module
station_coords = np.array([local_coords[s] for s in active_stations])

print("\nStation coordinates (local m):")
for i, sta in enumerate(active_stations):
    print(f"  {sta}: x={station_coords[i,0]:+.0f}, "
          f"y={station_coords[i,1]:+.0f}, z={station_coords[i,2]:+.0f}")

In [ ]:
# --- Travel-time location ---
# Use the best association event's picks to build observed arrival times
if labels is not None and n_events > 0:
    # Collect one pick per station from the best cluster
    obs_times = []
    obs_stations = []
    for idx in best_indices:
        sta_idx = pick_station_order[idx]
        sta_name = reverse_station_distance[sta_idx]
        if sta_name in active_stations and sta_name not in obs_stations:
            pick_time = UTCDateTime(num2date(pick_times[idx]))
            obs_times.append(float(pick_time - best_origin_time))
            obs_stations.append(sta_name)

    if len(obs_stations) >= 3:
        obs_times_arr = np.array(obs_times)
        obs_coords = np.array([
            station_coords[active_stations.index(s)] for s in obs_stations
        ])

        # Grid search bounds: ±50 km horizontal, 0-20 km depth
        tt_result = swe.location.locate_by_travel_time(
            obs_times_arr, obs_coords, velocity_model=3500.0,
            grid_bounds=((-50000, 50000), (-50000, 50000), (0, 20000)),
            grid_spacing=(500, 500, 500),
        )
        print(f"Travel-time location: x={tt_result.x:.0f} m, "
              f"y={tt_result.y:.0f} m, z={tt_result.z:.0f} m")
        print(f"Misfit: {tt_result.misfit:.4f}")
    else:
        tt_result = None
        print(f"Only {len(obs_stations)} stations with picks - need ≥3 for location.")

    # --- Amplitude-decay location ---
    # Compute maximum amplitudes per station for the event window
    obs_amps = []
    amp_stations = []
    for sta in active_stations:
        st_sta = st.select(station=sta)
        if st_sta:
            tr_event = st_sta[0].copy().trim(
                best_origin_time - 1.0, best_origin_time + 10.0,
            )
            if tr_event.stats.npts > 0:
                obs_amps.append(swe.utils.max_amplitude(tr_event))
                amp_stations.append(sta)

    if len(amp_stations) >= 3:
        obs_amps_arr = np.array(obs_amps)
        amp_coords = np.array([
            station_coords[active_stations.index(s)] for s in amp_stations
        ])
        amp_result = swe.location.amplocate(
            obs_amps_arr, amp_coords,
            grid_bounds=((-50000, 50000), (-50000, 50000), (0, 20000)),
            grid_spacing=(1000, 1000, 1000),
        )
        print(f"\nAmplitude location:   x={amp_result.x:.0f} m, "
              f"y={amp_result.y:.0f} m, z={amp_result.z:.0f} m")
        print(f"Misfit: {amp_result.misfit:.4f}")
    else:
        amp_result = None
        print(f"Only {len(amp_stations)} stations - need ≥3 for amplitude location.")
else:
    tt_result = None
    amp_result = None
    print("No association results - skipping location.")

In [ ]:
# Station network map with location estimates
fig, ax = plt.subplots(figsize=(8, 8))

# Plot stations
ax.scatter(
    station_coords[:, 0] / 1000, station_coords[:, 1] / 1000,
    marker="^", s=120, c="navy", zorder=5, label="Stations",
)
for i, sta in enumerate(active_stations):
    ax.annotate(
        sta, (station_coords[i, 0] / 1000, station_coords[i, 1] / 1000),
        textcoords="offset points", xytext=(6, 6), fontsize=9,
    )

# Plot location estimates
if tt_result is not None:
    ax.scatter(
        tt_result.x / 1000, tt_result.y / 1000,
        marker="*", s=300, c="red", zorder=6, edgecolors="k",
        label=f"Travel-time (z={tt_result.z/1000:.1f} km)",
    )
if amp_result is not None:
    ax.scatter(
        amp_result.x / 1000, amp_result.y / 1000,
        marker="*", s=300, c="orange", zorder=6, edgecolors="k",
        label=f"Amplitude (z={amp_result.z/1000:.1f} km)",
    )

ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
ax.set_title("Station network and event locations")
ax.legend()
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Step | Module | Key Function | Result |
|------|--------|-------------|--------|
| Data download | `swe.io` | `get_waveforms()` | MiniSEED files + in-memory Stream |
| Visualization | `swe.viz` | `plot_stream()`, `plot_spectrogram()`, `plot_multi_spectra()` | Waveform, spectrogram, and spectral figures |
| Noise detection | `swe.noise` | `find_noise_windows()` | Quiescent windows (few expected post-mainshock) |
| Phase picking | `swe.picking` | `predict_phase_arrivals()`, `pick_stream()` | P/S picks with confidence scores |
| Association | `swe.association` | `associate_picks()` | Picks clustered into events |
| Detection | EQcorrscan `Tribe` | `tribe.detect()` | Additional events via cross-correlation |
| Location | `swe.location` | `locate_by_travel_time()`, `amplocate()` | Hypocenter estimates (x, y, z) |

### Adapting for other datasets

To apply this pipeline to a different site, modify the configuration cell:

```python
# Example: adapting to a short-period / borehole network
NETWORK = "XX"
STATIONS = ["STA01", "STA02", "STA03", ...]
CHANNELS = "DP1"  # short-period, higher freq
START = UTCDateTime("2020-01-01T00:00:00")
END = UTCDateTime("2020-01-01T02:00:00")
```

Key adjustments:
- **STA/LTA parameters**: Scale with sampling rate (e.g., 250 Hz DP channels need STA~125, LTA~1250)
- **Picker model**: PhaseNet works best for broadband data; consider EQTransformer for short-period
- **Association thresholds**: Tighter `temporal_threshold` for closely-spaced networks
- **Velocity model**: Adjust `velocity_model` for local geology (3500 m/s = generic upper crust)